# Survey Question Retrieval Demo

This notebook demonstrates the current retrieval and reranking pipeline implemented in the application code.

It uses:
- OpenAI embeddings for semantic retrieval
- ChromaDB for persistent vector storage
- a cross-encoder reranker for second-stage ranking


## Setup

Before running the notebook, make sure:

1. The required packages are installed.
2. `OPENAI_API_KEY` is set in your environment.
3. You run the notebook from the repository context so imports from `app` work correctly.


In [1]:
from __future__ import annotations

import json
import os
import time
from pathlib import Path

import chromadb
from openai import OpenAI
from sentence_transformers import CrossEncoder

from app.models.schemas import SearchFilters
from app.services.lib import (
    EMBEDDING_MODEL,
    build_embedding_text,
    create_chroma_collection,
    embed_texts,
    prepare_data_for_chroma,
)
from app.services.retrieval import configure_retrieval, retrieve
from app.services.reranking import configure_reranker, rerank


/opt/anaconda3/envs/test/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
BASE_DIR = Path.cwd()

if BASE_DIR.name == "app":
    PROJECT_ROOT = BASE_DIR.parent
else:
    PROJECT_ROOT = BASE_DIR

DATA_PATH = PROJECT_ROOT / "data" / "profiling_questions.json"
CHROMA_PERSIST_DIR = PROJECT_ROOT / "app" / "chroma_db"
COLLECTION_NAME = "survey_questions"
RERANK_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"


## Load or create the vector collection

This mirrors the startup logic in `app/main.py`. If the Chroma collection already exists, we reuse it. Otherwise we build it from the source dataset.


In [3]:
def load_or_create_collection(client: OpenAI, chroma_client: chromadb.PersistentClient):
    try:
        collection = chroma_client.get_collection(name=COLLECTION_NAME)
        print(f"Loaded existing ChromaDB collection: {COLLECTION_NAME}")
        return collection
    except Exception:
        print(f"Collection '{COLLECTION_NAME}' not found. Building it from source data.")

    with DATA_PATH.open() as f:
        questions = json.load(f)

    print(f"Loaded {len(questions)} questions from {DATA_PATH}")

    document_texts = [build_embedding_text(q) for q in questions]
    document_embeddings = embed_texts(client, document_texts)

    collection = chroma_client.create_collection(
        name=COLLECTION_NAME,
        metadata={"description": "Survey profiling questions with semantic search"},
    )

    ids, documents, embeddings, metadatas = prepare_data_for_chroma(
        questions,
        document_texts,
        document_embeddings,
    )
    create_chroma_collection(collection, ids, documents, embeddings, metadatas)
    print(f"Created ChromaDB collection: {COLLECTION_NAME}")
    return collection


In [4]:
client = OpenAI()
chroma_client = chromadb.PersistentClient(path=str(CHROMA_PERSIST_DIR))
reranker_model = CrossEncoder(RERANK_MODEL)
collection = load_or_create_collection(client, chroma_client)

configure_retrieval(client=client, collection=collection)
configure_reranker(reranker_model=reranker_model)

print(f"Collection count: {collection.count()}")


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 11834.94it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded existing ChromaDB collection: survey_questions
Collection count: 300


## Helper for displaying results


In [5]:
def print_results(results: list[dict], score_key: str = "retrieval_score") -> None:
    for rank, result in enumerate(results, start=1):
        print(f"Rank {rank} | {result['question_id']} | {score_key}={result.get(score_key, 0):.4f}")
        print(result['text'])
        print(f"category={result['category']} | subcategory={result['subcategory']} | demographic_focus={result['demographic_focus']}")
        if 'rerank_score' in result:
            print(f"retrieval_score={result['retrieval_score']:.4f} | rerank_score={result['rerank_score']:.4f}")
        print()


## Retrieval-only example


In [6]:
query = "We are targeting health-conscious millennials in urban areas who are interested in sustainable living and plant-based diets."
results, retrieval_time_ms = retrieve(query=query, top_k=5)
print(f"Retrieval time: {retrieval_time_ms:.2f} ms\n")
print_results(results)


Retrieval time: 2654.00 ms

Rank 1 | q_069 | retrieval_score=0.5119
How often do you consume plant-based meat or dairy alternatives?
category=Health & Wellness | subcategory=Nutrition | demographic_focus=gen_z

Rank 2 | q_288 | retrieval_score=0.4990
Which of the following have you reduced or eliminated for environmental reasons?
category=Environmental Attitudes | subcategory=Lifestyle Changes | demographic_focus=millennials

Rank 3 | q_075 | retrieval_score=0.4749
Which factors most influence your choice of fitness routine or gym?
category=Health & Wellness | subcategory=Fitness | demographic_focus=millennials

Rank 4 | q_271 | retrieval_score=0.4650
Do you actively track or try to reduce your personal carbon footprint?
category=Environmental Attitudes | subcategory=Carbon Footprint | demographic_focus=millennials

Rank 5 | q_062 | retrieval_score=0.4648
Which mental health or wellness apps have you used in the past year?
category=Health & Wellness | subcategory=Mental Health | demogr

## Retrieval with metadata filters


In [7]:
filters = SearchFilters(category="Financial Behavior")
filtered_results, filtered_time_ms = retrieve(
    query="Targeting young professionals interested in fintech and mobile banking",
    top_k=5,
    filters=filters,
)
print(f"Filtered retrieval time: {filtered_time_ms:.2f} ms\n")
print_results(filtered_results)


Filtered retrieval time: 177.34 ms

Rank 1 | q_041 | retrieval_score=0.5214
Do you currently use any automated savings or round-up investment tools?
category=Financial Behavior | subcategory=Fintech Tools | demographic_focus=millennials

Rank 2 | q_036 | retrieval_score=0.4949
How often do you check your bank account or financial app balances?
category=Financial Behavior | subcategory=Digital Banking | demographic_focus=millennials

Rank 3 | q_054 | retrieval_score=0.4905
What factors would convince you to switch to a fully digital, branchless bank?
category=Financial Behavior | subcategory=Digital Banking | demographic_focus=general

Rank 4 | q_045 | retrieval_score=0.4790
Rank your trust level across financial institution types: traditional banks, online-only banks, credit unions, fintech apps, investment platforms.
category=Financial Behavior | subcategory=Banking Preferences | demographic_focus=general

Rank 5 | q_039 | retrieval_score=0.4679
Rank the following factors when choosin

## Retrieval plus reranking

We first over-retrieve a larger candidate set, then use the cross-encoder to improve the final ordering.


In [8]:
candidates, retrieval_time_ms = retrieve(
    query="We are targeting parents concerned about children's screen time and digital wellbeing",
    top_k=5,
    retrieval_multiplier=3,  # Retrieve more candidates for reranking
)
reranked_results, rerank_time_ms = rerank(
    query="We are targeting parents concerned about children's screen time and digital wellbeing",
    candidates=candidates,
    top_k=5,
)
print(f"Retrieval time: {retrieval_time_ms:.2f} ms")
print(f"Rerank time: {rerank_time_ms:.2f} ms\n")
print_results(candidates)
print_results(reranked_results, score_key="rerank_score")


Retrieval time: 2060.77 ms
Rerank time: 320.85 ms

Rank 1 | q_115 | retrieval_score=0.6263
What is your primary concern when it comes to children's use of technology?
category=Technology Usage | subcategory=Digital Wellness | demographic_focus=parents
retrieval_score=0.6263 | rerank_score=-6.1614

Rank 2 | q_080 | retrieval_score=0.6225
How concerned are you about the long-term health effects of screen time?
category=Health & Wellness | subcategory=Digital Wellness | demographic_focus=parents
retrieval_score=0.6225 | rerank_score=-3.5415

Rank 3 | q_091 | retrieval_score=0.4685
How concerned are you about the privacy of your personal data online?
category=Technology Usage | subcategory=Privacy & Security | demographic_focus=general
retrieval_score=0.4685 | rerank_score=-10.5726

Rank 4 | q_086 | retrieval_score=0.4578
How many hours per day do you spend on your smartphone?
category=Technology Usage | subcategory=Mobile | demographic_focus=general
retrieval_score=0.4578 | rerank_score=-

## Notes

This notebook is intended as a quick demonstration artifact for the assignment.

It shows:
- how the collection is initialized
- retrieval-only behavior
- metadata filtering
- second-stage reranking
